In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

In [60]:
RANDOM_STATE = 42
TARGET = "employed_status"
ID_COL = "anonymised_id"
N_SPLITS = 5

train = pd.read_csv("data/train.csv")
train = train.dropna(subset=["employed_status"])
test = pd.read_csv("data/test.csv")
groups_train = train[ID_COL]

Feature Engineering: Tenure, gated by prior employment and age x employed_lag and work_readiness_score x is_first_round

In [61]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["has_history"] = df["lag_round"].notna().astype(int)
    df["employed_lag_num"] = df["employed_lag"]  # 0/1/NaN
    df["employed_lag_x_recency"] = df["employed_lag_num"].fillna(0) * df["has_history"]

    # --- Tenure, gated by prior employment ---------------------------------
    # only treat tenure as a risk signal for rows that WERE employed last
    # round -- zero it out otherwise, so the coefficient isn't diluted by
    # unrelated non-employed zeros.
    df["tenure_lag_missing"] = df["tenure_lag"].isna().astype(int)
    df["tenure_lag"] = df["tenure_lag"].fillna(0)
    df["log_tenure_lag"] = np.log1p(df["tenure_lag"].clip(lower=0))
    df["log_tenure_lag_if_employed"] = df["log_tenure_lag"] * df["employed_lag_num"].fillna(0)

    df["is_first_round"] = (df["total_historical_rounds"] <= 1).astype(int)

     # --- FIX: Center age before squaring ---------------------------------
    df["age"] = df["age"].fillna(df["age"].median())
    age_mean = df["age"].mean()  # Calculate mean after imputation
    df["age_centered"] = df["age"] - age_mean  # ← NEW
    df["age_sq"] = df["age_centered"] ** 2  # ← CHANGED: square centered age

    # --- age x employed_lag --------------------------------------------
    # Age likely means something different depending on prior state: among
    # the already-employed it's closer to a tenure/seniority proxy; among
    # the not-employed it's closer to a "how long searching" proxy.
    df["age_x_employed_lag"] = df["age"] * df["employed_lag_num"].fillna(0)

    # --- work_readiness_score x is_first_round --------------------------
    # Purpose-built forward-looking score should matter most when it's the
    # ONLY forward signal available (no employed_lag / status history).
    df["work_readiness_x_first_round"] = df["work_readiness_score"].fillna(
        df["work_readiness_score"].median()
    ) * df["is_first_round"]

    return df

In [62]:
def add_seasonality_features(df: pd.DataFrame, date_col: str = "survey_date") -> pd.DataFrame:
    """Add cyclical seasonality features: sin/cos of month."""
    df = df.copy()
    
    if date_col not in df.columns:
        df["month_sin"] = 0
        df["month_cos"] = 0
        df["month_sin_x_employed_lag"] = 0
        df["month_cos_x_employed_lag"] = 0
        return df
    
    # Extract month
    month = pd.to_datetime(df[date_col]).dt.month
    
    # Cyclical encoding
    df["month_sin"] = np.sin(2 * np.pi * month / 12)
    df["month_cos"] = np.cos(2 * np.pi * month / 12)
    
    # Interaction with employed_lag - season affects transitions differently
    df["month_sin_x_employed_lag"] = df["month_sin"] * df["employed_lag_num"].fillna(0)
    df["month_cos_x_employed_lag"] = df["month_cos"] * df["employed_lag_num"].fillna(0)
    
    return df

In [63]:
def make_interaction_categorical(df, col_a, col_b, new_col, min_count=None,
                                  train_ref=None, other_label="Other"):
    """Combine two categorical columns into one 'A||B' categorical.
    NaNs are stringified so 'Missing' combinations are preserved as their
    own category rather than dropped."""
    a = df[col_a].astype(str).fillna("Missing")
    b = df[col_b].astype(str).fillna("Missing")
    df[new_col] = a + "||" + b

    if min_count is not None:
        ref = train_ref if train_ref is not None else df
        counts = ref[new_col].value_counts()
        keep = set(counts[counts >= min_count].index)
        df[new_col] = df[new_col].where(df[new_col].isin(keep), other_label)
    return df

In [64]:
def add_frequency_encoding(train_df, test_df, col, new_col=None):
    new_col = new_col or f"{col}_freq"
    freq_map = train_df[col].value_counts(normalize=True)
    train_df[new_col] = train_df[col].map(freq_map).fillna(0)
    test_df[new_col] = test_df[col].map(freq_map).fillna(0)
    return train_df, test_df


def collapse_rare_categories(train_df, test_df, col, min_count=30, other_label="Other"):
    counts = train_df[col].value_counts()
    keep = set(counts[counts >= min_count].index)

    def _collapse(series):
        return series.where(series.isin(keep) | series.isna(), other_label)

    train_df[col] = _collapse(train_df[col])
    test_df[col] = _collapse(test_df[col])
    return train_df, test_df

def extract_month_from_date(df: pd.DataFrame, date_col: str = "survey_date") -> pd.Series:
    """Extract month from survey_date column."""
    if date_col not in df.columns:
        return pd.Series(np.nan, index=df.index)
    return pd.to_datetime(df[date_col]).dt.month

In [ ]:
# ---------------------------------------------------------------------------
# 1. New-entrant-specific features
# ---------------------------------------------------------------------------
def add_new_entrant_features(df: pd.DataFrame) -> pd.DataFrame:
    """Features that only make sense to lean on when there's no lag history
    to fall back on -- interactions are zeroed out for returning respondents
    so they don't dilute the signal the model already has for that group."""
    df = df.copy()

    is_new = (df["has_history"] == 0).astype(int)  # requires has_history already added

    # education_level x school_quintile: quality-of-schooling proxy that
    # only exists as a meaningful predictor pre-labor-market-entry.
    edu = df["education_level"].astype(str).fillna("Missing")
    quint = df["school_quintile"].astype(str).fillna("Missing")
    df["education_x_quintile"] = (edu + "||" + quint).where(is_new.astype(bool), "NA||NA")

    # education_field x province: field-specific labor demand varies
    # regionally -- e.g. a trades qualification may transfer very
    # differently in Gauteng vs a rural province.
    field = df["education_field"].astype(str).fillna("Missing")
    prov = df["province"].astype(str).fillna("Missing")
    df["field_x_province"] = (field + "||" + prov).where(is_new.astype(bool), "NA||NA")

    # matric subject composite: pure/physical science + math lit as a
    # rough numeracy signal, zeroed for returning respondents.
    matric_num = (
    pd.to_numeric(df["matric_mathpure"], errors="coerce").fillna(0)
    + pd.to_numeric(df["matric_physicalscience"], errors="coerce").fillna(0)
    + pd.to_numeric(df["matric_mathlit"], errors="coerce").fillna(0)
    )
    df["matric_numeracy_x_new"] = matric_num * is_new

    df["is_new_entrant"] = is_new
    return df

def add_conditional_rate_encoding(train_df, val_df, col, target_col,
                                   condition_col="has_history", condition_val=0,
                                   smoothing=20, new_col=None, n_splits=N_SPLITS):
    """Employment-rate encoding fit ONLY on rows matching `condition_col ==
    condition_val` (e.g. new entrants). Falls back to the global new-entrant
    mean for unseen categories/rows outside the condition, so returning
    respondents don't get a rate computed from a population they're not
    part of."""
    new_col = new_col or f"{col}_rate_new_entrant"
    subset = train_df[train_df[condition_col] == condition_val]
    global_mean = subset[target_col].mean()
 
    oof = pd.Series(index=subset.index, dtype=float)
    gkf = GroupKFold(n_splits=n_splits)
    groups = subset[ID_COL]
    for tr_idx, v_idx in gkf.split(subset, subset[target_col], groups):
        fold_tr = subset.iloc[tr_idx]
        stats = fold_tr.groupby(col)[target_col].agg(["mean", "count"])
        smoothed = (stats["mean"] * stats["count"] + global_mean * smoothing) / (
            stats["count"] + smoothing
        )
        val_keys = subset.iloc[v_idx][col]
        oof.iloc[v_idx] = val_keys.map(smoothed).fillna(global_mean).values
 
    full_stats = subset.groupby(col)[target_col].agg(["mean", "count"])
    full_smoothed = (full_stats["mean"] * full_stats["count"] + global_mean * smoothing) / (
        full_stats["count"] + smoothing
    )
 
    train_df[new_col] = global_mean             # rows outside the condition
    train_df.loc[subset.index, new_col] = oof    # OOF for rows inside it
    val_df[new_col] = val_df[col].map(full_smoothed).fillna(global_mean)
    return train_df, val_df


In [67]:
train = engineer_features(train)
test = engineer_features(test)

train = add_seasonality_features(train)
test = add_seasonality_features(test)


# --- combined interaction categoricals -------------------------------
train = make_interaction_categorical(train, "gender", "status_broad_lag",
                                      "gender_x_status_lag")
test = make_interaction_categorical(test, "gender", "status_broad_lag",
                                     "gender_x_status_lag")

train = make_interaction_categorical(train, "education_level", "status_broad_lag",
                                      "education_x_status_lag")
test = make_interaction_categorical(test, "education_level", "status_broad_lag",
                                     "education_x_status_lag")

# race x education_level: sparser combo, so collapse rare cells using
# TRAIN-only counts to avoid leakage.
train = make_interaction_categorical(train, "race", "education_level",
                                      "race_x_education", min_count=50,
                                      train_ref=train)
test = make_interaction_categorical(test, "race", "education_level",
                                     "race_x_education")

# map test's raw combos through the same keep-set as train (anything not
# seen with min_count in train becomes "Other")
_keep_race_edu = set(train["race_x_education"].unique()) - {"Other"}
test["race_x_education"] = test["race_x_education"].where(
    test["race_x_education"].isin(_keep_race_edu), "Other"
)




In [ ]:

# ---------------------------------------------------------------------------
# 2. Round-8 walk-forward cutoff: build train/val
# ---------------------------------------------------------------------------
# Assumes `train` already has engineer_features / add_seasonality_features
# applied, matching your existing pipeline.
train = add_new_entrant_features(train)
test = add_new_entrant_features(test)


numeric_features = [
    "age", "age_sq", "employed_lag_x_recency", "log_tenure_lag_if_employed",
    "tenure_lag_missing", "total_historical_rounds", "has_history",
    "is_first_round", "work_readiness_score", "work_readiness_x_first_round",
    "age_x_employed_lag", "municipality_freq", "month_sin", "month_cos",
    "month_sin_x_employed_lag", "month_cos_x_employed_lag",
    "matric_numeracy_x_new",                       # NEW
    "municipality_rate_new_entrant",                # NEW
    "province_rate_new_entrant",                    # NEW
]
categorical_features = [
    "status_broad_lag", "gender", "race", "province", "education_level",
    "gender_x_status_lag", "education_x_status_lag",
    "education_x_quintile",     # NEW
    "field_x_province",         # NEW
]


_always_keep = {"municipality_rate_new_entrant", "province_rate_new_entrant"}
numeric_features = [c for c in numeric_features if c in train.columns or c in _always_keep]
categorical_features = [c for c in categorical_features if c in train.columns]


PIPELINE

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

#Boosting
categorical_pipeline = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", drop="if_binary", sparse_output=False)),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
])



model = HistGradientBoostingClassifier(
    max_iter=1000,              # number of boosting rounds (trees)
    learning_rate=0.02,
    max_depth=4,               # shallow trees — this is the point of boosting
    min_samples_leaf=50,
    l2_regularization=1.0,
    early_stopping=True,       # holds out a slice internally to stop overfitting
    n_iter_no_change=20,        # a bit more patience since lr is lower
    validation_fraction=0.15,
    random_state=RANDOM_STATE,
)

pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", model),
])

Cross-Validation

In [ ]:
from sklearn.metrics import roc_auc_score

# Use several trailing rounds as successive cutoffs instead of one,
# to average out the "small round" noise problem.
rounds_sorted = sorted(train["current_round"].unique())
val_rounds = rounds_sorted[-3:]  # last 3 rounds as walk-forward cutoffs

fold_aucs = []
subgroup_aucs = []  # track has_history split per fold

for cutoff in val_rounds:
    tr_df = train[train["current_round"] < cutoff].copy()
    val_df = train[train["current_round"] == cutoff].copy()

    if val_df.empty or tr_df.empty:
        continue

    # all target/frequency encodings fit on tr_df ONLY (time-safe)
    tr_df, val_df = add_frequency_encoding(tr_df, val_df, "municipality")
    tr_df, val_df = collapse_rare_categories(tr_df, val_df, "education_level", min_count=100)
    
    
    tr_df, val_df = add_conditional_rate_encoding(
        tr_df, val_df, "municipality", TARGET, condition_col="has_history", condition_val=0
    )
    tr_df, val_df = add_conditional_rate_encoding(
        tr_df, val_df, "province", TARGET, condition_col="has_history", condition_val=0
    )

    X_tr = tr_df[numeric_features + categorical_features]
    X_val = val_df[numeric_features + categorical_features]

    y_tr = tr_df[TARGET].astype(int)
    y_val = val_df[TARGET].astype(int)

    pipeline.fit(X_tr, y_tr)
    val_probs = pipeline.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, val_probs)
    fold_aucs.append(auc)

    # subgroup check: new entrants vs returning respondents
    has_hist = val_df["has_history"].values.astype(bool)
    sub = {}
    if has_hist.sum() > 20:
        sub["returning"] = roc_auc_score(y_val[has_hist], val_probs[has_hist])
    if (~has_hist).sum() > 20:
        sub["new_entrant"] = roc_auc_score(y_val[~has_hist], val_probs[~has_hist])
    subgroup_aucs.append(sub)

    print(f"Cutoff round {cutoff}: n_val={len(val_df)}, AUC={auc:.5f}, subgroups={sub}")

print(f"\nWalk-forward mean AUC: {np.mean(fold_aucs):.5f} (+/- {np.std(fold_aucs):.5f})")

Cutoff round 6: n_val=4883, AUC=0.59410, subgroups={'returning': 0.6790888247282609, 'new_entrant': 0.5599456600017398}
Cutoff round 7: n_val=3199, AUC=0.62809, subgroups={'returning': 0.6395594209758213, 'new_entrant': 0.6189447096672346}
Cutoff round 8: n_val=2333, AUC=0.63987, subgroups={'returning': 0.7556935817805384, 'new_entrant': 0.6101773596226259}

Walk-forward mean AUC: 0.62069 (+/- 0.01940)


Fit the pipeline on the full training data and make predictions on the test set:

In [ ]:
train, test = add_frequency_encoding(train, test, "municipality")
train, test = collapse_rare_categories(train, test, "education_level", min_count=100)

train, test = add_conditional_rate_encoding(
    train, test, "municipality", TARGET, condition_col="has_history", condition_val=0
)
train, test = add_conditional_rate_encoding(
    train, test, "province", TARGET, condition_col="has_history", condition_val=0
)

y_train = train[TARGET].astype(int)   # NEW: was never defined in this notebook

X_train_final = train[numeric_features + categorical_features]
X_test_final = test[numeric_features + categorical_features]
pipeline.fit(X_train_final, y_train)
test_probs = pipeline.predict_proba(X_test_final)[:, 1]

submission = pd.DataFrame({
    ID_COL: test[ID_COL],
    "employed_prob": test_probs,
})
submission.to_csv("Submissions/Boosting.csv", index=False)
print("\nSaved Boosting.csv")


Saved Boosting.csv
